In [1]:
import tensorflow as tf
import numpy as np
import os
import collections
import gdown
import tensorflow_federated as tff
import matplotlib.pyplot as plt
from tensorflow_federated.python.learning.optimizers import sgdm

2026-06-26 18:21:22.901814: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-26 18:21:23.549897: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-26 18:21:23.549956: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-26 18:21:23.554861: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-26 18:21:23.834111: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-26 18:21:23.834968: I tensorflow/core/platform/cpu_feature_guard.cc:182] This Tens

In [24]:
# Function to load TFRecord dataset
def load_tfrecord_dataset(file_path):
    raw_dataset = tf.data.TFRecordDataset(file_path)

    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.int64),
        "video_key": tf.io.FixedLenFeature([], tf.string),
    }

    def _parse_function(proto):
        example = tf.io.parse_single_example(proto, feature_description)

        image = tf.io.decode_raw(example["image"], tf.uint8)
        image = tf.cast(image, tf.float32) / 255.0
        image = tf.reshape(image, (784,))

        return {
            "x": image,
            "y": example["label"]
        }

    parsed_dataset = raw_dataset.map(_parse_function)

    return parsed_dataset

In [25]:
# Function to create federated data for all clients
def create_federated_data(client_ids, data_dir):
    federated_data = []
    for client_id in client_ids:
        # Construct the file path for each client
        train_file = os.path.join(data_dir, f"{client_id}.tfrecord")

        # Check if the file exists before loading
        if not os.path.exists(train_file):
            print(f"File not found: {train_file}")
            continue

        train_dataset = load_tfrecord_dataset(train_file)
        # Add a batching step to create 2D input tensors
        train_dataset = train_dataset.batch(10)
        federated_data.append(train_dataset)

    return federated_data

In [26]:
# Download TFRecord files from Google Drive folder
# Replace this link with your Google Drive TFRecords folder link.
# Folder must contain: client_1.tfrecord, client_2.tfrecord, client_3.tfrecord, client_4.tfrecord, client_5.tfrecord

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/16c-TMBXRLMW8aO0gjIaEB6HVtOaR7mnF?usp=drive_link"

# Keep same path as original notebook so remaining code does not need changes
# LOCAL_TFRECORD_DIR = "/content/drive/MyDrive/Pratyush/Experiment Suspicious Activity Detection/Experiment Data Creation/TF Records"
# os.makedirs(LOCAL_TFRECORD_DIR, exist_ok=True)

gdown.download_folder(
    url=DRIVE_FOLDER_URL,
    # output=LOCAL_TFRECORD_DIR,
    quiet=False,
    use_cookies=False
)

print("Downloaded files:")
# print(os.listdir(LOCAL_TFRECORD_DIR))


Retrieving folder contents


Processing file 1kL_aB1-FDMaI8FdAiMO1K3jgQ5lqjhqc client_1.tfrecord
Processing file 13CxRxEgSP14Pdngbn0lPwS5P0neK3VeD client_2.tfrecord
Processing file 1EKrg1Daj6ItOlgWzYuDeMMGFMQ95mRFr client_3.tfrecord
Processing file 1UM3UYh0o8HpJrPfjpyVR1qowweagHzaM client_4.tfrecord
Processing file 1z0Mw71aewsLzeApj4F2wKEZbtwDUFaoM client_5.tfrecord
Processing file 1CYVP-C09aRlZFl1jDQPR_e2NoAISKNCG test.tfrecord


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1kL_aB1-FDMaI8FdAiMO1K3jgQ5lqjhqc
From (redirected): https://drive.google.com/uc?id=1kL_aB1-FDMaI8FdAiMO1K3jgQ5lqjhqc&confirm=t&uuid=41cc919e-4b7f-4c22-87d5-8d13f7bfb5cb
To: /workspaces/tff-notebook/TFRecords/client_1.tfrecord
100%|██████████| 582M/582M [00:09<00:00, 60.9MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=13CxRxEgSP14Pdngbn0lPwS5P0neK3VeD
From (redirected): https://drive.google.com/uc?id=13CxRxEgSP14Pdngbn0lPwS5P0neK3VeD&confirm=t&uuid=18631fc3-3744-4cb2-95e1-d6f4bbcc0ab4
To: /workspaces/tff-notebook/TFRecords/client_2.tfrecord
100%|██████████| 289M/289M [00:07<00:00, 39.2MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1EKrg1Daj6ItOlgWzYuDeMMGFMQ95mRFr
To: /workspaces/tff-notebook/TFRecords/client_3.tfrecord
100%|██████████| 67.3M/67.3M [00:01<00:00, 49.3MB/s]
Downloading...


Downloaded files:


Download completed


In [27]:
# Set the correct directory paths for the train data
train_data_dir = "/workspaces/tff-notebook/TFRecords"

# Generate federated training data
client_ids = [f'client_{i}' for i in range(1, 6)]
federated_train_data = create_federated_data(client_ids, train_data_dir)

In [28]:
def create_keras_model():
    # Create a Sequential model
    model = tf.keras.Sequential([
        # Input layer
        tf.keras.layers.InputLayer(input_shape=(784,)),

        # First hidden layer with reduced units
        tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),

        # Second hidden layer with fewer units
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),

        # Third hidden layer with Batch Normalization and Dropout
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        # Fourth hidden layer, further reduced units
        tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        # Fifth hidden layer with minimal units
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        # Output layer with softmax activation for multi-class classification
        tf.keras.layers.Dense(14, activation='softmax')
    ])

    return model

In [29]:
# Function to build the TFF model from Keras model
def model_fn():
    keras_model = create_keras_model()
    # Check if federated_train_data is not empty
    if federated_train_data:
        # Use tff.learning.models.from_keras_model instead of tff.learning.from_keras_model
        return tff.learning.models.from_keras_model(
            keras_model,
            input_spec=federated_train_data[0].element_spec,  # Use the first client's dataset spec
            loss=tf.keras.losses.SparseCategoricalCrossentropy(),
            metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
        )
    else:
        # Handle the case where no client data is loaded
        print("Error: No client data loaded. Please check the data directory and client IDs.")
        return None

In [30]:
def preprocess(dataset):
    # Function to preprocess each element in the dataset
    def element_fn(element):
        # Clip the label value to be within the range [0, 9]
        label = element['y'] - 1
        label = tf.clip_by_value(label, 0, 9)

        return collections.OrderedDict([
            ('x', element['x']),
            ('y', label),
        ])
    return dataset.map(element_fn)

# Assuming federated_train_data is your existing dataset
federated_train_data = [preprocess(client_data) for client_data in federated_train_data]

In [31]:
# Use the unweighted FedAvg function
iterative_process = tff.learning.algorithms.build_unweighted_fed_avg(
    model_fn=model_fn,
    client_optimizer_fn=tff.learning.optimizers.build_sgdm(learning_rate=0.02),
    server_optimizer_fn=tff.learning.optimizers.build_sgdm(learning_rate=1.0),
    metrics_aggregator=tff.learning.metrics.sum_then_finalize,
)

/workspaces/tff-notebook/.venv/lib/python3.9/site-packages/tensorflow_federated/python/learning/models/keras_utils.py:201: UserWarning: Batch Normalization contains non-trainable variables that won't be updated during the training. Consider using Group Normalization instead.
  warnings.warn(


In [32]:
'''
iterative_process = tff.learning.algorithms.build_weighted_fed_avg(
    model_fn=model_fn,
    client_optimizer_fn=tff.learning.optimizers.build_sgdm(learning_rate=0.02),
    server_optimizer_fn=tff.learning.optimizers.build_sgdm(learning_rate=1.0)
)
'''

'\niterative_process = tff.learning.algorithms.build_weighted_fed_avg(\n    model_fn=model_fn,\n    client_optimizer_fn=tff.learning.optimizers.build_sgdm(learning_rate=0.02),\n    server_optimizer_fn=tff.learning.optimizers.build_sgdm(learning_rate=1.0)\n)\n'

In [16]:
print(iterative_process.initialize.type_signature.formatted_representation())

( -> <
  global_model_weights=<
    trainable=<
      float32[784,256],
      float32[256],
      float32[256],
      float32[256],
      float32[256,128],
      float32[128],
      float32[128],
      float32[128],
      float32[128,64],
      float32[64],
      float32[64],
      float32[64],
      float32[64,32],
      float32[32],
      float32[32],
      float32[32],
      float32[32,16],
      float32[16],
      float32[16],
      float32[16],
      float32[16,14],
      float32[14]
    >,
    non_trainable=<
      float32[256],
      float32[256],
      float32[128],
      float32[128],
      float32[64],
      float32[64],
      float32[32],
      float32[32],
      float32[16],
      float32[16]
    >
  >,
  distributor=<>,
  client_work=<
    learning_rate=float32
  >,
  aggregator=<
    <>,
    <>
  >,
  finalizer=<
    learning_rate=float32
  >
>@SERVER)


In [33]:
# Initialize the state
state = iterative_process.initialize()

2026-06-26 18:36:34.512632: I tensorflow/core/grappler/devices.cc:66] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
2026-06-26 18:36:34.512848: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-06-26 18:36:34.562730: I tensorflow/core/grappler/devices.cc:66] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
2026-06-26 18:36:34.562870: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-06-26 18:36:34.564674: I tensorflow/core/grappler/devices.cc:66] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
2026-06-26 18:36:34.564774: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session


In [35]:
# Extract and print the global model weights
global_model_weights = iterative_process.get_model_weights(state)
print("\nGlobal Model Weights:")
for weight in global_model_weights.trainable:
    print(f"Weight Shape: {weight.shape}")
    print(f"Weight Matrix:\n{weight}\n")


Global Model Weights:
Weight Shape: (784, 256)
Weight Matrix:
[[ 0.0231277  -0.00852124 -0.02518695 ...  0.00775776  0.01666372
  -0.07155008]
 [ 0.03895859 -0.05934728 -0.04886223 ...  0.03886234 -0.02386463
   0.00383071]
 [-0.00175609  0.0429826  -0.06031521 ... -0.0726377   0.05691564
   0.06726715]
 ...
 [ 0.03019255  0.02809639 -0.00012803 ...  0.07346277 -0.072537
   0.05680932]
 [-0.00742604 -0.04483439  0.03928409 ... -0.06729683 -0.06601865
   0.01700319]
 [ 0.00959095 -0.03054751 -0.05103119 ...  0.05821541  0.0396772
   0.05445664]]

Weight Shape: (256,)
Weight Matrix:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 

In [36]:
# Lists to store metrics for each round
accuracy_values = []
loss_values = []
round_numbers = []
initial_global_weights = state.global_model_weights.trainable
previous_global_weights = initial_global_weights

In [ ]:
# Open the file for writing updates
with open("global_model_updates_full.txt", "a") as file:

    # Log the initialized model weights
    file.write("\n" + "+"*100 + "\n")
    file.write("Initialized Model Weights:\n")
    for idx, layer in enumerate(initial_global_weights):
        file.write(f"  Layer {idx + 1} - Weight (Layer size: {layer.shape}):\n")
        file.write(f"{np.array2string(layer, threshold=np.inf, separator=', ')}\n")

    # Training loop
    for round_num in range(1, 51):
        # Perform one round of training
        state, metrics = iterative_process.next(state, federated_train_data)
        print(f"Round {round_num}, Metrics={metrics}")

        # Extract accuracy and loss
        accuracy = metrics['client_work']['train']['sparse_categorical_accuracy']
        loss = metrics['client_work']['train']['loss']

        # Track metrics
        accuracy_values.append(accuracy)
        loss_values.append(loss)
        round_numbers.append(round_num)

        # Extract current global weights
        current_global_weights = state.global_model_weights.trainable

        # Log global updates
        file.write("\n" + "+"*100 + "\n")
        file.write(f"Round {round_num} - Global Model Updates:\n")
        for idx, (prev_layer, curr_layer) in enumerate(zip(previous_global_weights, current_global_weights)):
            layer_update = curr_layer - prev_layer  # Calculate the difference
            file.write(f"  Layer {idx + 1} - Update (Layer size: {curr_layer.shape}):\n")
            file.write(f"{np.array2string(layer_update, threshold=np.inf, separator=', ')}\n")

        # Update previous weights for the next round
        previous_global_weights = current_global_weights

2026-06-26 18:37:04.568852: I tensorflow/core/grappler/devices.cc:66] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
2026-06-26 18:37:04.569024: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-06-26 18:37:04.861038: I tensorflow/core/grappler/devices.cc:66] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
2026-06-26 18:37:04.861171: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-06-26 18:37:04.866927: I tensorflow/core/grappler/devices.cc:66] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
2026-06-26 18:37:04.867029: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-06-26 18:37:04.874807: I tensorflow/core/grappler/devices.cc:66] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
2026-06-26 18:37:04.874904: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session


Round 1, Metrics=OrderedDict([('distributor', ()), ('client_work', OrderedDict([('train', OrderedDict([('sparse_categorical_accuracy', 0.8998528), ('loss', 0.36425003), ('num_examples', 1268892), ('num_batches', 126891)]))])), ('aggregator', OrderedDict([('mean_value', ()), ('mean_count', ())])), ('finalizer', OrderedDict([('update_non_finite', 0)]))])
Round 2, Metrics=OrderedDict([('distributor', ()), ('client_work', OrderedDict([('train', OrderedDict([('sparse_categorical_accuracy', 0.89282936), ('loss', 0.37956324), ('num_examples', 1268892), ('num_batches', 126891)]))])), ('aggregator', OrderedDict([('mean_value', ()), ('mean_count', ())])), ('finalizer', OrderedDict([('update_non_finite', 0)]))])
Round 3, Metrics=OrderedDict([('distributor', ()), ('client_work', OrderedDict([('train', OrderedDict([('sparse_categorical_accuracy', 0.89221543), ('loss', 0.3805107), ('num_examples', 1268892), ('num_batches', 126891)]))])), ('aggregator', OrderedDict([('mean_value', ()), ('mean_count',